In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName("first selftest")\
        .getOrCreate()

In [3]:
df = spark.read.csv("testdata1.csv", header = True, inferSchema = True )

df.show()
df.printSchema()

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  1|Alice Johnson|Engineering| 95000|
|  2|    Bob Smith|  Marketing| 72000|
|  3|  Carol White|Engineering|105000|
|  4|  David Brown|         HR| 58000|
|  5| Eva Martinez|  Marketing| 81000|
|  6|    Frank Lee|Engineering|112000|
|  7|    Grace Kim|    Finance| 88000|
|  8| Henry Wilson|         HR| 61000|
|  9|  Isabel Chen|    Finance| 94000|
| 10| James Taylor|Engineering| 99000|
| 11|  Karen Davis|  Marketing| 76000|
| 12|Liam Anderson|    Finance| 82000|
+---+-------------+-----------+------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), nullable = True),
    StructField("name", StringType(), nullable = True),
    StructField("department", StringType(), nullable = True),
    StructField("salary", IntegerType(), nullable = True),
])

df2 = spark.read.csv("testdata1.csv", header= True, schema = schema)

df2.show(5)          # First 10 rows, formatted as table
df2.printSchema()     # Column names + data types
df2.describe().show() # Count, mean, stddev, min, max for numeric cols
df2.count()    

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  1|Alice Johnson|Engineering| 95000|
|  2|    Bob Smith|  Marketing| 72000|
|  3|  Carol White|Engineering|105000|
|  4|  David Brown|         HR| 58000|
|  5| Eva Martinez|  Marketing| 81000|
+---+-------------+-----------+------+
only showing top 5 rows

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)

+-------+-----------------+-------------+-----------+------------------+
|summary|               id|         name| department|            salary|
+-------+-----------------+-------------+-----------+------------------+
|  count|               12|           12|         12|                12|
|   mean|              6.5|         NULL|       NULL|           85250.0|
| stddev|3.605551275463989|         NULL|       NULL|16771.864969211223|
|    min|                1|A

12

In [5]:
df2.select("department").show()

+-----------+
| department|
+-----------+
|Engineering|
|  Marketing|
|Engineering|
|         HR|
|  Marketing|
|Engineering|
|    Finance|
|         HR|
|    Finance|
|Engineering|
|  Marketing|
|    Finance|
+-----------+



In [6]:
from pyspark.sql.functions import col

In [7]:
df2.select(col("salary"), col("department")).show()

+------+-----------+
|salary| department|
+------+-----------+
| 95000|Engineering|
| 72000|  Marketing|
|105000|Engineering|
| 58000|         HR|
| 81000|  Marketing|
|112000|Engineering|
| 88000|    Finance|
| 61000|         HR|
| 94000|    Finance|
| 99000|Engineering|
| 76000|  Marketing|
| 82000|    Finance|
+------+-----------+



In [8]:
df2.select("department", (col("salary")*12). alias("annual salary")).show()

+-----------+-------------+
| department|annual salary|
+-----------+-------------+
|Engineering|      1140000|
|  Marketing|       864000|
|Engineering|      1260000|
|         HR|       696000|
|  Marketing|       972000|
|Engineering|      1344000|
|    Finance|      1056000|
|         HR|       732000|
|    Finance|      1128000|
|Engineering|      1188000|
|  Marketing|       912000|
|    Finance|       984000|
+-----------+-------------+



In [9]:
df2.filter("salary>80000").show()

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  1|Alice Johnson|Engineering| 95000|
|  3|  Carol White|Engineering|105000|
|  5| Eva Martinez|  Marketing| 81000|
|  6|    Frank Lee|Engineering|112000|
|  7|    Grace Kim|    Finance| 88000|
|  9|  Isabel Chen|    Finance| 94000|
| 10| James Taylor|Engineering| 99000|
| 12|Liam Anderson|    Finance| 82000|
+---+-------------+-----------+------+



In [10]:
df2.where((col("salary")>82000) | (col("department")=="Finance")).show()

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  1|Alice Johnson|Engineering| 95000|
|  3|  Carol White|Engineering|105000|
|  6|    Frank Lee|Engineering|112000|
|  7|    Grace Kim|    Finance| 88000|
|  9|  Isabel Chen|    Finance| 94000|
| 10| James Taylor|Engineering| 99000|
| 12|Liam Anderson|    Finance| 82000|
+---+-------------+-----------+------+



In [11]:
df2.where(~(col("department")=="Engineering")).show()

+---+-------------+----------+------+
| id|         name|department|salary|
+---+-------------+----------+------+
|  2|    Bob Smith| Marketing| 72000|
|  4|  David Brown|        HR| 58000|
|  5| Eva Martinez| Marketing| 81000|
|  7|    Grace Kim|   Finance| 88000|
|  8| Henry Wilson|        HR| 61000|
|  9|  Isabel Chen|   Finance| 94000|
| 11|  Karen Davis| Marketing| 76000|
| 12|Liam Anderson|   Finance| 82000|
+---+-------------+----------+------+



In [12]:
df2.filter(col("name").startswith("A")).show()

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  1|Alice Johnson|Engineering| 95000|
+---+-------------+-----------+------+



In [13]:
df2.filter(col("name").contains("sab")).show()

+---+-----------+----------+------+
| id|       name|department|salary|
+---+-----------+----------+------+
|  9|Isabel Chen|   Finance| 94000|
+---+-----------+----------+------+



In [14]:
df2.filter((col("department")). isin("Finance", "HR")).show()

+---+-------------+----------+------+
| id|         name|department|salary|
+---+-------------+----------+------+
|  4|  David Brown|        HR| 58000|
|  7|    Grace Kim|   Finance| 88000|
|  8| Henry Wilson|        HR| 61000|
|  9|  Isabel Chen|   Finance| 94000|
| 12|Liam Anderson|   Finance| 82000|
+---+-------------+----------+------+



In [15]:
df2.withColumn("bonus", col("salary")*0.5).show()

+---+-------------+-----------+------+-------+
| id|         name| department|salary|  bonus|
+---+-------------+-----------+------+-------+
|  1|Alice Johnson|Engineering| 95000|47500.0|
|  2|    Bob Smith|  Marketing| 72000|36000.0|
|  3|  Carol White|Engineering|105000|52500.0|
|  4|  David Brown|         HR| 58000|29000.0|
|  5| Eva Martinez|  Marketing| 81000|40500.0|
|  6|    Frank Lee|Engineering|112000|56000.0|
|  7|    Grace Kim|    Finance| 88000|44000.0|
|  8| Henry Wilson|         HR| 61000|30500.0|
|  9|  Isabel Chen|    Finance| 94000|47000.0|
| 10| James Taylor|Engineering| 99000|49500.0|
| 11|  Karen Davis|  Marketing| 76000|38000.0|
| 12|Liam Anderson|    Finance| 82000|41000.0|
+---+-------------+-----------+------+-------+



In [16]:
from pyspark.sql.functions import lit

In [17]:
df2.withColumn("country", lit("Uganda")).show()

+---+-------------+-----------+------+-------+
| id|         name| department|salary|country|
+---+-------------+-----------+------+-------+
|  1|Alice Johnson|Engineering| 95000| Uganda|
|  2|    Bob Smith|  Marketing| 72000| Uganda|
|  3|  Carol White|Engineering|105000| Uganda|
|  4|  David Brown|         HR| 58000| Uganda|
|  5| Eva Martinez|  Marketing| 81000| Uganda|
|  6|    Frank Lee|Engineering|112000| Uganda|
|  7|    Grace Kim|    Finance| 88000| Uganda|
|  8| Henry Wilson|         HR| 61000| Uganda|
|  9|  Isabel Chen|    Finance| 94000| Uganda|
| 10| James Taylor|Engineering| 99000| Uganda|
| 11|  Karen Davis|  Marketing| 76000| Uganda|
| 12|Liam Anderson|    Finance| 82000| Uganda|
+---+-------------+-----------+------+-------+



In [18]:
df2.withColumn("is_Active", lit(True)).show()

+---+-------------+-----------+------+---------+
| id|         name| department|salary|is_Active|
+---+-------------+-----------+------+---------+
|  1|Alice Johnson|Engineering| 95000|     true|
|  2|    Bob Smith|  Marketing| 72000|     true|
|  3|  Carol White|Engineering|105000|     true|
|  4|  David Brown|         HR| 58000|     true|
|  5| Eva Martinez|  Marketing| 81000|     true|
|  6|    Frank Lee|Engineering|112000|     true|
|  7|    Grace Kim|    Finance| 88000|     true|
|  8| Henry Wilson|         HR| 61000|     true|
|  9|  Isabel Chen|    Finance| 94000|     true|
| 10| James Taylor|Engineering| 99000|     true|
| 11|  Karen Davis|  Marketing| 76000|     true|
| 12|Liam Anderson|    Finance| 82000|     true|
+---+-------------+-----------+------+---------+



In [19]:
df2_enriched = (
    df2
    .withColumn("bonus", col("salary")*0.3)
    .withColumn("totalPay", col("bonus") + col("salary"))
    .withColumn("taxes", col("bonus")*0.2)
)

df2_enriched.show()

+---+-------------+-----------+------+-------+--------+------+
| id|         name| department|salary|  bonus|totalPay| taxes|
+---+-------------+-----------+------+-------+--------+------+
|  1|Alice Johnson|Engineering| 95000|28500.0|123500.0|5700.0|
|  2|    Bob Smith|  Marketing| 72000|21600.0| 93600.0|4320.0|
|  3|  Carol White|Engineering|105000|31500.0|136500.0|6300.0|
|  4|  David Brown|         HR| 58000|17400.0| 75400.0|3480.0|
|  5| Eva Martinez|  Marketing| 81000|24300.0|105300.0|4860.0|
|  6|    Frank Lee|Engineering|112000|33600.0|145600.0|6720.0|
|  7|    Grace Kim|    Finance| 88000|26400.0|114400.0|5280.0|
|  8| Henry Wilson|         HR| 61000|18300.0| 79300.0|3660.0|
|  9|  Isabel Chen|    Finance| 94000|28200.0|122200.0|5640.0|
| 10| James Taylor|Engineering| 99000|29700.0|128700.0|5940.0|
| 11|  Karen Davis|  Marketing| 76000|22800.0| 98800.0|4560.0|
| 12|Liam Anderson|    Finance| 82000|24600.0|106600.0|4920.0|
+---+-------------+-----------+------+-------+--------+

In [20]:
df2_enriched.drop("salary").show()

+---+-------------+-----------+-------+--------+------+
| id|         name| department|  bonus|totalPay| taxes|
+---+-------------+-----------+-------+--------+------+
|  1|Alice Johnson|Engineering|28500.0|123500.0|5700.0|
|  2|    Bob Smith|  Marketing|21600.0| 93600.0|4320.0|
|  3|  Carol White|Engineering|31500.0|136500.0|6300.0|
|  4|  David Brown|         HR|17400.0| 75400.0|3480.0|
|  5| Eva Martinez|  Marketing|24300.0|105300.0|4860.0|
|  6|    Frank Lee|Engineering|33600.0|145600.0|6720.0|
|  7|    Grace Kim|    Finance|26400.0|114400.0|5280.0|
|  8| Henry Wilson|         HR|18300.0| 79300.0|3660.0|
|  9|  Isabel Chen|    Finance|28200.0|122200.0|5640.0|
| 10| James Taylor|Engineering|29700.0|128700.0|5940.0|
| 11|  Karen Davis|  Marketing|22800.0| 98800.0|4560.0|
| 12|Liam Anderson|    Finance|24600.0|106600.0|4920.0|
+---+-------------+-----------+-------+--------+------+



In [21]:
df2.select(
    col("name"),
    (col("salary") *0.3) .alias("bonus added"),
    (col("salary") * 12) .alias("annual salary")
).show()

+-------------+-----------+-------------+
|         name|bonus added|annual salary|
+-------------+-----------+-------------+
|Alice Johnson|    28500.0|      1140000|
|    Bob Smith|    21600.0|       864000|
|  Carol White|    31500.0|      1260000|
|  David Brown|    17400.0|       696000|
| Eva Martinez|    24300.0|       972000|
|    Frank Lee|    33600.0|      1344000|
|    Grace Kim|    26400.0|      1056000|
| Henry Wilson|    18300.0|       732000|
|  Isabel Chen|    28200.0|      1128000|
| James Taylor|    29700.0|      1188000|
|  Karen Davis|    22800.0|       912000|
|Liam Anderson|    24600.0|       984000|
+-------------+-----------+-------------+



In [22]:
df2_enriched.show()

+---+-------------+-----------+------+-------+--------+------+
| id|         name| department|salary|  bonus|totalPay| taxes|
+---+-------------+-----------+------+-------+--------+------+
|  1|Alice Johnson|Engineering| 95000|28500.0|123500.0|5700.0|
|  2|    Bob Smith|  Marketing| 72000|21600.0| 93600.0|4320.0|
|  3|  Carol White|Engineering|105000|31500.0|136500.0|6300.0|
|  4|  David Brown|         HR| 58000|17400.0| 75400.0|3480.0|
|  5| Eva Martinez|  Marketing| 81000|24300.0|105300.0|4860.0|
|  6|    Frank Lee|Engineering|112000|33600.0|145600.0|6720.0|
|  7|    Grace Kim|    Finance| 88000|26400.0|114400.0|5280.0|
|  8| Henry Wilson|         HR| 61000|18300.0| 79300.0|3660.0|
|  9|  Isabel Chen|    Finance| 94000|28200.0|122200.0|5640.0|
| 10| James Taylor|Engineering| 99000|29700.0|128700.0|5940.0|
| 11|  Karen Davis|  Marketing| 76000|22800.0| 98800.0|4560.0|
| 12|Liam Anderson|    Finance| 82000|24600.0|106600.0|4920.0|
+---+-------------+-----------+------+-------+--------+

In [23]:
df2_enriched.withColumnRenamed("name", "employee_name").show()

+---+-------------+-----------+------+-------+--------+------+
| id|employee_name| department|salary|  bonus|totalPay| taxes|
+---+-------------+-----------+------+-------+--------+------+
|  1|Alice Johnson|Engineering| 95000|28500.0|123500.0|5700.0|
|  2|    Bob Smith|  Marketing| 72000|21600.0| 93600.0|4320.0|
|  3|  Carol White|Engineering|105000|31500.0|136500.0|6300.0|
|  4|  David Brown|         HR| 58000|17400.0| 75400.0|3480.0|
|  5| Eva Martinez|  Marketing| 81000|24300.0|105300.0|4860.0|
|  6|    Frank Lee|Engineering|112000|33600.0|145600.0|6720.0|
|  7|    Grace Kim|    Finance| 88000|26400.0|114400.0|5280.0|
|  8| Henry Wilson|         HR| 61000|18300.0| 79300.0|3660.0|
|  9|  Isabel Chen|    Finance| 94000|28200.0|122200.0|5640.0|
| 10| James Taylor|Engineering| 99000|29700.0|128700.0|5940.0|
| 11|  Karen Davis|  Marketing| 76000|22800.0| 98800.0|4560.0|
| 12|Liam Anderson|    Finance| 82000|24600.0|106600.0|4920.0|
+---+-------------+-----------+------+-------+--------+

In [24]:
from pyspark.sql.functions import when

In [25]:
df2.withColumn(
    "salary range",
    when(col("salary")>80000, "high😎")
    .otherwise("struggling😂")
    
).show()

+---+-------------+-----------+------+------------+
| id|         name| department|salary|salary range|
+---+-------------+-----------+------+------------+
|  1|Alice Johnson|Engineering| 95000|      high😎|
|  2|    Bob Smith|  Marketing| 72000|struggling😂|
|  3|  Carol White|Engineering|105000|      high😎|
|  4|  David Brown|         HR| 58000|struggling😂|
|  5| Eva Martinez|  Marketing| 81000|      high😎|
|  6|    Frank Lee|Engineering|112000|      high😎|
|  7|    Grace Kim|    Finance| 88000|      high😎|
|  8| Henry Wilson|         HR| 61000|struggling😂|
|  9|  Isabel Chen|    Finance| 94000|      high😎|
| 10| James Taylor|Engineering| 99000|      high😎|
| 11|  Karen Davis|  Marketing| 76000|struggling😂|
| 12|Liam Anderson|    Finance| 82000|      high😎|
+---+-------------+-----------+------+------------+



In [26]:
df2.withColumn(
    "Rating",
    when(col("salary")>100000, "premium🤩🤩")
    .when(col("salary")>80000, "not bad😎😎")
    .otherwise("struggler🙄🙄")
).show()

+---+-------------+-----------+------+-------------+
| id|         name| department|salary|       Rating|
+---+-------------+-----------+------+-------------+
|  1|Alice Johnson|Engineering| 95000|  not bad😎😎|
|  2|    Bob Smith|  Marketing| 72000|struggler🙄🙄|
|  3|  Carol White|Engineering|105000|  premium🤩🤩|
|  4|  David Brown|         HR| 58000|struggler🙄🙄|
|  5| Eva Martinez|  Marketing| 81000|  not bad😎😎|
|  6|    Frank Lee|Engineering|112000|  premium🤩🤩|
|  7|    Grace Kim|    Finance| 88000|  not bad😎😎|
|  8| Henry Wilson|         HR| 61000|struggler🙄🙄|
|  9|  Isabel Chen|    Finance| 94000|  not bad😎😎|
| 10| James Taylor|Engineering| 99000|  not bad😎😎|
| 11|  Karen Davis|  Marketing| 76000|struggler🙄🙄|
| 12|Liam Anderson|    Finance| 82000|  not bad😎😎|
+---+-------------+-----------+------+-------------+



In [27]:
df2.withColumn(
    "title",
    when((col("department")== "Engineering")& (col("salary") > 100000), "Senior Engineer")
    .when((col("department")== "Engineering")& (col("salary") < 100000), "Junior Engineer")
    .otherwise("Other")
).show()

+---+-------------+-----------+------+---------------+
| id|         name| department|salary|          title|
+---+-------------+-----------+------+---------------+
|  1|Alice Johnson|Engineering| 95000|Junior Engineer|
|  2|    Bob Smith|  Marketing| 72000|          Other|
|  3|  Carol White|Engineering|105000|Senior Engineer|
|  4|  David Brown|         HR| 58000|          Other|
|  5| Eva Martinez|  Marketing| 81000|          Other|
|  6|    Frank Lee|Engineering|112000|Senior Engineer|
|  7|    Grace Kim|    Finance| 88000|          Other|
|  8| Henry Wilson|         HR| 61000|          Other|
|  9|  Isabel Chen|    Finance| 94000|          Other|
| 10| James Taylor|Engineering| 99000|Junior Engineer|
| 11|  Karen Davis|  Marketing| 76000|          Other|
| 12|Liam Anderson|    Finance| 82000|          Other|
+---+-------------+-----------+------+---------------+



In [28]:
df2.withColumn("bonus", 
    when(col("salary") > 82000, col("salary") * 0.15)
    .otherwise(col("salary") * lit(0.05))
).show()

+---+-------------+-----------+------+-------+
| id|         name| department|salary|  bonus|
+---+-------------+-----------+------+-------+
|  1|Alice Johnson|Engineering| 95000|14250.0|
|  2|    Bob Smith|  Marketing| 72000| 3600.0|
|  3|  Carol White|Engineering|105000|15750.0|
|  4|  David Brown|         HR| 58000| 2900.0|
|  5| Eva Martinez|  Marketing| 81000| 4050.0|
|  6|    Frank Lee|Engineering|112000|16800.0|
|  7|    Grace Kim|    Finance| 88000|13200.0|
|  8| Henry Wilson|         HR| 61000| 3050.0|
|  9|  Isabel Chen|    Finance| 94000|14100.0|
| 10| James Taylor|Engineering| 99000|14850.0|
| 11|  Karen Davis|  Marketing| 76000| 3800.0|
| 12|Liam Anderson|    Finance| 82000| 4100.0|
+---+-------------+-----------+------+-------+



In [29]:
from pyspark.sql.functions import col, when, lit, expr

In [30]:
result = (
    df2
    .withColumn(
        "salary_category",
        when(col("salary")>90000, "damn🔥")
        .when(col("salary")>75000, "good")
        .otherwise("oh_mehn😂")
    )
    .withColumn(
        "bonus_rate",
        when(col("salary_category")== "damn", 0.15)
        .when(col("salary_category")== "good", 0.10)
        .otherwise(0.05)
    )
    .withColumn("bonus_amount",col("salary")* col("bonus_rate"))
    .withColumn("total_comp", col("salary")+ col("bonus_amount"))
    .withColumn(
        "comp_status",
        when(col("total_comp")>90000, "Top_Earner")
        .otherwise("Average_Earner")
    )
    .withColumn("experience_level", 
                when((col("salary")>98000) & (col("department")=="Engineering"), "Senior")
                .when((col("salary")>68000) & (col("department")=="Engineering"), "Junior")
                .otherwise("N/A")

               )
)

result.show()

+---+-------------+-----------+------+---------------+----------+------------+----------+--------------+----------------+
| id|         name| department|salary|salary_category|bonus_rate|bonus_amount|total_comp|   comp_status|experience_level|
+---+-------------+-----------+------+---------------+----------+------------+----------+--------------+----------------+
|  1|Alice Johnson|Engineering| 95000|         damn🔥|      0.05|      4750.0|   99750.0|    Top_Earner|          Junior|
|  2|    Bob Smith|  Marketing| 72000|      oh_mehn😂|      0.05|      3600.0|   75600.0|Average_Earner|             N/A|
|  3|  Carol White|Engineering|105000|         damn🔥|      0.05|      5250.0|  110250.0|    Top_Earner|          Senior|
|  4|  David Brown|         HR| 58000|      oh_mehn😂|      0.05|      2900.0|   60900.0|Average_Earner|             N/A|
|  5| Eva Martinez|  Marketing| 81000|           good|       0.1|      8100.0|   89100.0|Average_Earner|             N/A|
|  6|    Frank Lee|Engineeri

In [31]:
from pyspark.sql.functions import avg,sum,count,max,min,round
df2.groupby("department").count().show()
result.groupby("comp_status")\
    .agg(
    max("salary").alias("max_salary"),
    round(avg("salary"),2).alias("avg_salary")
    ).show()

+-----------+-----+
| department|count|
+-----------+-----+
|Engineering|    4|
|         HR|    2|
|    Finance|    3|
|  Marketing|    3|
+-----------+-----+

+--------------+----------+----------+
|   comp_status|max_salary|avg_salary|
+--------------+----------+----------+
|Average_Earner|     81000|   69600.0|
|    Top_Earner|    112000|  96428.57|
+--------------+----------+----------+



In [32]:
df2.groupBy("department").agg(
        avg("salary").alias("avg_salary"),
        max("salary").alias("max_salary"),
        min("salary").alias("min_salary"),
        count("*").alias("headcount")
    ).show()

+-----------+-----------------+----------+----------+---------+
| department|       avg_salary|max_salary|min_salary|headcount|
+-----------+-----------------+----------+----------+---------+
|Engineering|         102750.0|    112000|     95000|        4|
|         HR|          59500.0|     61000|     58000|        2|
|    Finance|          88000.0|     94000|     82000|        3|
|  Marketing|76333.33333333333|     81000|     72000|        3|
+-----------+-----------------+----------+----------+---------+



In [33]:
data = [
(1, "John", "IT", "Kampala", "Full-Time", 80000, "AI"),
(2, "Anna", "IT", "Kampala", "Full-Time", 90000, "AI"),
(3, "Mike", "HR", "Nairobi", "Part-Time", 40000, "Recruitment"),
(4, "Lisa", "HR", "Nairobi", "Full-Time", 45000, "Training"),
(5, "David", "Finance", "Kigali", "Full-Time", 70000, "Audit"),
(6, "Grace", "Finance", "Kigali", "Contract", 72000, "Audit"),
(7, "Paul", "Sales", "Kampala", "Full-Time", 60000, "Retail"),
(8, "Sarah", "Sales", "Nairobi", "Part-Time", 55000, "Wholesale"),
(9, "John", "IT", "Kampala", "Full-Time", 80000, "Cloud"),
(10, "Emma", "IT", "Nairobi", "Contract", 85000, "Cloud"),
(11, "Brian", "Finance", "Kigali", "Full-Time", 75000, "Tax"),
(12, "Alice", "Sales", "Kampala", "Full-Time", 62000, "Retail"),
(13, "Tom", "IT", "Kigali", "Part-Time", 50000, "Support"),
(14, "Nancy", "HR", "Kampala", "Contract", 38000, "Recruitment"),
(15, "Kevin", "IT", "Nairobi", "Full-Time", 95000, "AI"),
(16, "Sophia", "Sales", "Nairobi", "Full-Time", 58000, "Retail"),
(17, "Mark", "Finance", "Kampala", "Full-Time", 68000, "Tax"),
(18, "Lucy", "HR", "Kigali", "Part-Time", 42000, "Training"),
(19, "Daniel", "IT", "Kigali", "Full-Time", 88000, "Cloud"),
(20, "Olivia", "Sales", "Kampala", "Contract", 53000, "Wholesale")
]
columns = ["emp_id", "name", "department", "location", "comp_status", "salary", "project"]

df3 = spark.createDataFrame(data, columns)

df3.show()

+------+------+----------+--------+-----------+------+-----------+
|emp_id|  name|department|location|comp_status|salary|    project|
+------+------+----------+--------+-----------+------+-----------+
|     1|  John|        IT| Kampala|  Full-Time| 80000|         AI|
|     2|  Anna|        IT| Kampala|  Full-Time| 90000|         AI|
|     3|  Mike|        HR| Nairobi|  Part-Time| 40000|Recruitment|
|     4|  Lisa|        HR| Nairobi|  Full-Time| 45000|   Training|
|     5| David|   Finance|  Kigali|  Full-Time| 70000|      Audit|
|     6| Grace|   Finance|  Kigali|   Contract| 72000|      Audit|
|     7|  Paul|     Sales| Kampala|  Full-Time| 60000|     Retail|
|     8| Sarah|     Sales| Nairobi|  Part-Time| 55000|  Wholesale|
|     9|  John|        IT| Kampala|  Full-Time| 80000|      Cloud|
|    10|  Emma|        IT| Nairobi|   Contract| 85000|      Cloud|
|    11| Brian|   Finance|  Kigali|  Full-Time| 75000|        Tax|
|    12| Alice|     Sales| Kampala|  Full-Time| 62000|     Ret

In [34]:
from pyspark.sql.functions import (sum,count,min,max,avg,countDistinct,collect_list,collect_set,stddev,variance,first,last)

In [35]:
(
    df3.groupBy("department")
    .agg(
        countDistinct("name").alias("unique_names"),
        count("*").alias("total_per_dep"),
        sum("salary").alias("total_payroll"),
        max("salary").alias("max_pay"),
        min("salary").alias("min_pay"),
        avg("salary").alias("avg_salary"),
    )
    .filter(col("avg_salary") > 60000)
    .show()
)

+----------+------------+-------------+-------------+-------+-------+-----------------+
|department|unique_names|total_per_dep|total_payroll|max_pay|min_pay|       avg_salary|
+----------+------------+-------------+-------------+-------+-------+-----------------+
|   Finance|           4|            4|       285000|  75000|  68000|          71250.0|
|        IT|           6|            7|       568000|  95000|  50000|81142.85714285714|
+----------+------------+-------------+-------------+-------+-------+-----------------+



In [36]:
df3.groupBy("department").agg(
    collect_list("location").alias("place_list"),
    collect_set("location").alias("place_set"),
    stddev("salary").alias("salary_deviation"),
    variance("salary").alias("salary_variance")
).show(truncate=False)

+----------+-------------------------------------------------------------+--------------------------+------------------+--------------------+
|department|place_list                                                   |place_set                 |salary_deviation  |salary_variance     |
+----------+-------------------------------------------------------------+--------------------------+------------------+--------------------+
|HR        |[Nairobi, Nairobi, Kampala, Kigali]                          |[Kampala, Kigali, Nairobi]|2986.0788111948195|8916666.666666666   |
|Finance   |[Kigali, Kigali, Kigali, Kampala]                            |[Kampala, Kigali]         |2986.0788111948177|8916666.666666657   |
|IT        |[Kampala, Kampala, Kampala, Nairobi, Kigali, Nairobi, Kigali]|[Kampala, Kigali, Nairobi]|14747.07396320336 |2.1747619047619048E8|
|Sales     |[Kampala, Nairobi, Kampala, Nairobi, Kampala]                |[Kampala, Nairobi]        |3646.916505762094 |1.33E7              |
+-----

In [37]:
df3.groupBy("department").agg(
    first("salary").alias("first_salary"),
    last("salary").alias("last_salary")
).show()

+----------+------------+-----------+
|department|first_salary|last_salary|
+----------+------------+-----------+
|        HR|       40000|      42000|
|   Finance|       70000|      68000|
|        IT|       80000|      88000|
|     Sales|       60000|      53000|
+----------+------------+-----------+



In [38]:
df3.agg(
    count("*").alias("total_employees"),
    sum("salary").alias("total_payroll"),
    avg("salary").alias("company_avg_salary")
).show()

+---------------+-------------+------------------+
|total_employees|total_payroll|company_avg_salary|
+---------------+-------------+------------------+
|             20|      1306000|           65300.0|
+---------------+-------------+------------------+



In [39]:
(
    df3
    .groupBy("department", "location")
    .agg(
        avg("salary").alias("avg_salary"),
        count("*").alias("headcount")
    )
    .orderBy("department", "location")
    .show()
)

+----------+--------+------------------+---------+
|department|location|        avg_salary|headcount|
+----------+--------+------------------+---------+
|   Finance| Kampala|           68000.0|        1|
|   Finance|  Kigali| 72333.33333333333|        3|
|        HR| Kampala|           38000.0|        1|
|        HR|  Kigali|           42000.0|        1|
|        HR| Nairobi|           42500.0|        2|
|        IT| Kampala| 83333.33333333333|        3|
|        IT|  Kigali|           69000.0|        2|
|        IT| Nairobi|           90000.0|        2|
|     Sales| Kampala|58333.333333333336|        3|
|     Sales| Nairobi|           56500.0|        2|
+----------+--------+------------------+---------+



In [40]:
from pyspark.sql.functions import round, when

df3.groupBy("department").agg(
    round(avg("salary"), 2).alias("avg_salary"),
    round(stddev("salary"), 2).alias("salary_stddev")
).show()

+----------+----------+-------------+
|department|avg_salary|salary_stddev|
+----------+----------+-------------+
|        HR|   41250.0|      2986.08|
|   Finance|   71250.0|      2986.08|
|        IT|  81142.86|     14747.07|
|     Sales|   57600.0|      3646.92|
+----------+----------+-------------+



In [41]:
department_summary = (
    df3
    .withColumn(
               "salary_band",
               when(col("salary")> 65000, "senior")
                .otherwise("junior")
               )
    .groupBy("department")
    .agg(
        count("*").alias("total_staff"),
        round(avg("salary"),0).alias("avg_salary"),
        min("salary").alias("min_salary"),
        max("salary").alias("max_salary"),
        round(sum("salary"),0).alias("pay_roll"),
        countDistinct("location").alias("num_of_offices"),
        count(when(col("salary_band")=="senior",True)).alias("senior_count")
    )
)
department_summary.show()

+----------+-----------+----------+----------+----------+--------+--------------+------------+
|department|total_staff|avg_salary|min_salary|max_salary|pay_roll|num_of_offices|senior_count|
+----------+-----------+----------+----------+----------+--------+--------------+------------+
|     Sales|          5|   57600.0|     53000|     62000|  288000|             2|           0|
|        HR|          4|   41250.0|     38000|     45000|  165000|             3|           0|
|   Finance|          4|   71250.0|     68000|     75000|  285000|             2|           4|
|        IT|          7|   81143.0|     50000|     95000|  568000|             3|           6|
+----------+-----------+----------+----------+----------+--------+--------------+------------+



In [42]:
df3.orderBy("salary").show()

+------+------+----------+--------+-----------+------+-----------+
|emp_id|  name|department|location|comp_status|salary|    project|
+------+------+----------+--------+-----------+------+-----------+
|    14| Nancy|        HR| Kampala|   Contract| 38000|Recruitment|
|     3|  Mike|        HR| Nairobi|  Part-Time| 40000|Recruitment|
|    18|  Lucy|        HR|  Kigali|  Part-Time| 42000|   Training|
|     4|  Lisa|        HR| Nairobi|  Full-Time| 45000|   Training|
|    13|   Tom|        IT|  Kigali|  Part-Time| 50000|    Support|
|    20|Olivia|     Sales| Kampala|   Contract| 53000|  Wholesale|
|     8| Sarah|     Sales| Nairobi|  Part-Time| 55000|  Wholesale|
|    16|Sophia|     Sales| Nairobi|  Full-Time| 58000|     Retail|
|     7|  Paul|     Sales| Kampala|  Full-Time| 60000|     Retail|
|    12| Alice|     Sales| Kampala|  Full-Time| 62000|     Retail|
|    17|  Mark|   Finance| Kampala|  Full-Time| 68000|        Tax|
|     5| David|   Finance|  Kigali|  Full-Time| 70000|      Au

In [43]:
df3.orderBy(col("salary").desc()).show()

+------+------+----------+--------+-----------+------+-----------+
|emp_id|  name|department|location|comp_status|salary|    project|
+------+------+----------+--------+-----------+------+-----------+
|    15| Kevin|        IT| Nairobi|  Full-Time| 95000|         AI|
|     2|  Anna|        IT| Kampala|  Full-Time| 90000|         AI|
|    19|Daniel|        IT|  Kigali|  Full-Time| 88000|      Cloud|
|    10|  Emma|        IT| Nairobi|   Contract| 85000|      Cloud|
|     1|  John|        IT| Kampala|  Full-Time| 80000|         AI|
|     9|  John|        IT| Kampala|  Full-Time| 80000|      Cloud|
|    11| Brian|   Finance|  Kigali|  Full-Time| 75000|        Tax|
|     6| Grace|   Finance|  Kigali|   Contract| 72000|      Audit|
|     5| David|   Finance|  Kigali|  Full-Time| 70000|      Audit|
|    17|  Mark|   Finance| Kampala|  Full-Time| 68000|        Tax|
|    12| Alice|     Sales| Kampala|  Full-Time| 62000|     Retail|
|     7|  Paul|     Sales| Kampala|  Full-Time| 60000|     Ret

In [44]:
df3.orderBy("department", col("salary").desc()).show()


+------+------+----------+--------+-----------+------+-----------+
|emp_id|  name|department|location|comp_status|salary|    project|
+------+------+----------+--------+-----------+------+-----------+
|    11| Brian|   Finance|  Kigali|  Full-Time| 75000|        Tax|
|     6| Grace|   Finance|  Kigali|   Contract| 72000|      Audit|
|     5| David|   Finance|  Kigali|  Full-Time| 70000|      Audit|
|    17|  Mark|   Finance| Kampala|  Full-Time| 68000|        Tax|
|     4|  Lisa|        HR| Nairobi|  Full-Time| 45000|   Training|
|    18|  Lucy|        HR|  Kigali|  Part-Time| 42000|   Training|
|     3|  Mike|        HR| Nairobi|  Part-Time| 40000|Recruitment|
|    14| Nancy|        HR| Kampala|   Contract| 38000|Recruitment|
|    15| Kevin|        IT| Nairobi|  Full-Time| 95000|         AI|
|     2|  Anna|        IT| Kampala|  Full-Time| 90000|         AI|
|    19|Daniel|        IT|  Kigali|  Full-Time| 88000|      Cloud|
|    10|  Emma|        IT| Nairobi|   Contract| 85000|      Cl

In [45]:
df3.orderBy(col("salary").asc_nulls_first()).show()

+------+------+----------+--------+-----------+------+-----------+
|emp_id|  name|department|location|comp_status|salary|    project|
+------+------+----------+--------+-----------+------+-----------+
|    14| Nancy|        HR| Kampala|   Contract| 38000|Recruitment|
|     3|  Mike|        HR| Nairobi|  Part-Time| 40000|Recruitment|
|    18|  Lucy|        HR|  Kigali|  Part-Time| 42000|   Training|
|     4|  Lisa|        HR| Nairobi|  Full-Time| 45000|   Training|
|    13|   Tom|        IT|  Kigali|  Part-Time| 50000|    Support|
|    20|Olivia|     Sales| Kampala|   Contract| 53000|  Wholesale|
|     8| Sarah|     Sales| Nairobi|  Part-Time| 55000|  Wholesale|
|    16|Sophia|     Sales| Nairobi|  Full-Time| 58000|     Retail|
|     7|  Paul|     Sales| Kampala|  Full-Time| 60000|     Retail|
|    12| Alice|     Sales| Kampala|  Full-Time| 62000|     Retail|
|    17|  Mark|   Finance| Kampala|  Full-Time| 68000|        Tax|
|     5| David|   Finance|  Kigali|  Full-Time| 70000|      Au

In [46]:
data = [
(1, "John", "IT", 80000, 5000, "2022-01-15"),
(2, "Anna", "IT", None, 4000, "2021-07-10"),
(3, "Mike", "HR", 40000, None, "2020-03-01"),
(4, None, "HR", 45000, 2000, None),
(5, "David", None, 70000, 3000, "2023-05-20"),
(6, "Grace", "Finance", None, None, "2022-09-30"),
(7, "Paul", "Sales", 60000, 2500, None),
(8, "Sarah", "Sales", 55000, None, "2021-11-11"),
(9, "Tom", "IT", None, None, None),
(10, None, None, None, None, None)
]

columns = ["emp_id", "name", "department", "salary", "bonus", "hire_date"]

df_nulls = spark.createDataFrame(data, columns)

df_nulls.show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     1| John|        IT| 80000| 5000|2022-01-15|
|     2| Anna|        IT|  NULL| 4000|2021-07-10|
|     3| Mike|        HR| 40000| NULL|2020-03-01|
|     4| NULL|        HR| 45000| 2000|      NULL|
|     5|David|      NULL| 70000| 3000|2023-05-20|
|     6|Grace|   Finance|  NULL| NULL|2022-09-30|
|     7| Paul|     Sales| 60000| 2500|      NULL|
|     8|Sarah|     Sales| 55000| NULL|2021-11-11|
|     9|  Tom|        IT|  NULL| NULL|      NULL|
|    10| NULL|      NULL|  NULL| NULL|      NULL|
+------+-----+----------+------+-----+----------+



In [47]:
df_nulls.filter(col("salary").isNull()).show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     2| Anna|        IT|  NULL| 4000|2021-07-10|
|     6|Grace|   Finance|  NULL| NULL|2022-09-30|
|     9|  Tom|        IT|  NULL| NULL|      NULL|
|    10| NULL|      NULL|  NULL| NULL|      NULL|
+------+-----+----------+------+-----+----------+



In [48]:
df_nulls.filter(col("salary").isNotNull()).show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     1| John|        IT| 80000| 5000|2022-01-15|
|     3| Mike|        HR| 40000| NULL|2020-03-01|
|     4| NULL|        HR| 45000| 2000|      NULL|
|     5|David|      NULL| 70000| 3000|2023-05-20|
|     7| Paul|     Sales| 60000| 2500|      NULL|
|     8|Sarah|     Sales| 55000| NULL|2021-11-11|
+------+-----+----------+------+-----+----------+



In [49]:
from pyspark.sql.functions import col, isnan, when, count

In [50]:
df_nulls.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_nulls.columns
]).show()

+------+----+----------+------+-----+---------+
|emp_id|name|department|salary|bonus|hire_date|
+------+----+----------+------+-----+---------+
|     0|   2|         2|     4|    5|        4|
+------+----+----------+------+-----+---------+



In [51]:
total_rows = df_nulls.count()

df_nulls.select([
    (count(when(col(c).isNull(), 1)) / total_rows * 100).alias(c)
    for c in df_nulls.columns
]).show()

+------+----+----------+------+-----+---------+
|emp_id|name|department|salary|bonus|hire_date|
+------+----+----------+------+-----+---------+
|   0.0|20.0|      20.0|  40.0| 50.0|     40.0|
+------+----+----------+------+-----+---------+



In [52]:
df_nulls.na.drop().show()

+------+----+----------+------+-----+----------+
|emp_id|name|department|salary|bonus| hire_date|
+------+----+----------+------+-----+----------+
|     1|John|        IT| 80000| 5000|2022-01-15|
+------+----+----------+------+-----+----------+



In [53]:
df_nulls.na.drop(how="all").show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     1| John|        IT| 80000| 5000|2022-01-15|
|     2| Anna|        IT|  NULL| 4000|2021-07-10|
|     3| Mike|        HR| 40000| NULL|2020-03-01|
|     4| NULL|        HR| 45000| 2000|      NULL|
|     5|David|      NULL| 70000| 3000|2023-05-20|
|     6|Grace|   Finance|  NULL| NULL|2022-09-30|
|     7| Paul|     Sales| 60000| 2500|      NULL|
|     8|Sarah|     Sales| 55000| NULL|2021-11-11|
|     9|  Tom|        IT|  NULL| NULL|      NULL|
|    10| NULL|      NULL|  NULL| NULL|      NULL|
+------+-----+----------+------+-----+----------+



In [54]:
df_nulls.na.drop(subset = ["department", "salary"]).show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     1| John|        IT| 80000| 5000|2022-01-15|
|     3| Mike|        HR| 40000| NULL|2020-03-01|
|     4| NULL|        HR| 45000| 2000|      NULL|
|     7| Paul|     Sales| 60000| 2500|      NULL|
|     8|Sarah|     Sales| 55000| NULL|2021-11-11|
+------+-----+----------+------+-----+----------+



In [55]:
df_nulls.na.drop(thresh=4).show()

+------+-----+----------+------+-----+----------+
|emp_id| name|department|salary|bonus| hire_date|
+------+-----+----------+------+-----+----------+
|     1| John|        IT| 80000| 5000|2022-01-15|
|     2| Anna|        IT|  NULL| 4000|2021-07-10|
|     3| Mike|        HR| 40000| NULL|2020-03-01|
|     4| NULL|        HR| 45000| 2000|      NULL|
|     5|David|      NULL| 70000| 3000|2023-05-20|
|     6|Grace|   Finance|  NULL| NULL|2022-09-30|
|     7| Paul|     Sales| 60000| 2500|      NULL|
|     8|Sarah|     Sales| 55000| NULL|2021-11-11|
+------+-----+----------+------+-----+----------+



In [56]:
df_nulls.na.fill({
    "name":"unknown",
    "department": "unassigned",
    "salary": 0,
    "bonus": 0,
    "hire_date": "1900-01-01"
}).show()

+------+-------+----------+------+-----+----------+
|emp_id|   name|department|salary|bonus| hire_date|
+------+-------+----------+------+-----+----------+
|     1|   John|        IT| 80000| 5000|2022-01-15|
|     2|   Anna|        IT|     0| 4000|2021-07-10|
|     3|   Mike|        HR| 40000|    0|2020-03-01|
|     4|unknown|        HR| 45000| 2000|1900-01-01|
|     5|  David|unassigned| 70000| 3000|2023-05-20|
|     6|  Grace|   Finance|     0|    0|2022-09-30|
|     7|   Paul|     Sales| 60000| 2500|1900-01-01|
|     8|  Sarah|     Sales| 55000|    0|2021-11-11|
|     9|    Tom|        IT|     0|    0|1900-01-01|
|    10|unknown|unassigned|     0|    0|1900-01-01|
+------+-------+----------+------+-----+----------+



In [57]:
dept_avg = df_nulls.groupBy("department").agg(
    avg("salary").alias("dept_avg_salary")
)

df_with_avg = df_nulls.join(dept_avg, on="department", how="left")

from pyspark.sql.functions import when, coalesce

df_filled = df_with_avg.withColumn(
    "salary",
    coalesce(col("salary"), col("dept_avg_salary"))
).drop("dept_avg_salary")

df_filled.show()

+----------+------+-----+-------+-----+----------+
|department|emp_id| name| salary|bonus| hire_date|
+----------+------+-----+-------+-----+----------+
|        IT|     1| John|80000.0| 5000|2022-01-15|
|        IT|     2| Anna|80000.0| 4000|2021-07-10|
|        HR|     3| Mike|40000.0| NULL|2020-03-01|
|        HR|     4| NULL|45000.0| 2000|      NULL|
|      NULL|     5|David|70000.0| 3000|2023-05-20|
|   Finance|     6|Grace|   NULL| NULL|2022-09-30|
|     Sales|     7| Paul|60000.0| 2500|      NULL|
|     Sales|     8|Sarah|55000.0| NULL|2021-11-11|
|      NULL|    10| NULL|   NULL| NULL|      NULL|
|        IT|     9|  Tom|80000.0| NULL|      NULL|
+----------+------+-----+-------+-----+----------+



In [58]:
# Remove fully duplicate rows (all columns must match)
df.distinct().show()

# Remove duplicates based on specific columns
df.dropDuplicates(["name", "department"]).show()

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  5| Eva Martinez|  Marketing| 81000|
|  6|    Frank Lee|Engineering|112000|
|  4|  David Brown|         HR| 58000|
|  1|Alice Johnson|Engineering| 95000|
|  9|  Isabel Chen|    Finance| 94000|
|  8| Henry Wilson|         HR| 61000|
| 10| James Taylor|Engineering| 99000|
|  7|    Grace Kim|    Finance| 88000|
|  3|  Carol White|Engineering|105000|
| 12|Liam Anderson|    Finance| 82000|
| 11|  Karen Davis|  Marketing| 76000|
|  2|    Bob Smith|  Marketing| 72000|
+---+-------------+-----------+------+

+---+-------------+-----------+------+
| id|         name| department|salary|
+---+-------------+-----------+------+
|  8| Henry Wilson|         HR| 61000|
|  6|    Frank Lee|Engineering|112000|
|  3|  Carol White|Engineering|105000|
|  7|    Grace Kim|    Finance| 88000|
|  1|Alice Johnson|Engineering| 95000|
|  5| Eva Martinez|  Marketing| 81000|
|  2|    Bob Smith|  Mar